# Instationäre Behälterleerung – numerische Lösung

**BIW3-10 Weiterführende Hydromechanik · Übung 3 · Aufgabe 5**

In der Übung haben wir die Behälterleerung analytisch gelöst (Trennung der Variablen):

$$A\,\frac{\mathrm{d}h}{\mathrm{d}t} = -\alpha A_{\mathrm{o}}\sqrt{2gh} \qquad\Longrightarrow\qquad h(t) = h_0\left(1-\frac{t}{T_E}\right)^2,\qquad T_E = \frac{2A\sqrt{h_0}}{\alpha A_{\mathrm{o}}\sqrt{2g}}$$

Hier lösen wir dieselbe DGL **numerisch** mit dem **expliziten Euler-Verfahren** und vergleichen mit der analytischen Lösung.

**Lernziele:** Euler-Verfahren implementieren · Schrittweitenfehler quantifizieren · Physik der Quasistationarität mit der Schrittweite verknüpfen.

## 1. Parameter (wie in Aufgabe 4)

Zylinderspeicher, scharfkantige Bodenöffnung, kein Zulauf.

In [ ]:
# Parameter
g     = 9.81      # Erdbeschleunigung [m/s²]
D     = 2.0       # Behälterdurchmesser [m]
d_o   = 0.08      # Öffnungsdurchmesser [m]
alpha = 0.62      # Ausflusszahl (Folien 16)
h0    = 3.6       # Anfangswasserstand [m]

In [ ]:
# Abgeleitete Größen
import numpy as np
import matplotlib.pyplot as plt

A  = np.pi * D**2 / 4          # Behälterquerschnitt [m²]
A_o = np.pi * d_o**2 / 4       # Öffnungsquerschnitt [m²]
kappa = alpha * A_o * np.sqrt(2*g) / A   # Koeffizient der DGL [m^0.5/s]

T_E = 2 * A * np.sqrt(h0) / (alpha * A_o * np.sqrt(2*g))  # analytische Auslaufzeit [s]
Q0  = alpha * A_o * np.sqrt(2*g*h0)                       # Anfangsabfluss [m³/s]

print(f"A       = {A:.4f} m²")
print(f"A_o     = {A_o*1e3:.3f} e-3 m²")
print(f"kappa   = {kappa:.6e} m^0.5/s")
print(f"T_E     = {T_E:.1f} s = {T_E/60:.2f} min")
print(f"Q_0     = {Q0*1000:.1f} L/s")

## 2. Analytische Lösung als Referenz

$h(t) = h_0\,(1-t/T_E)^2$ und $Q(t) = \alpha A_{\mathrm{o}}\sqrt{2g\,h(t)}$.

In [ ]:
def h_analytisch(t):
    return h0 * (1 - t/T_E)**2

def Q_analytisch(t):
    return alpha * A_o * np.sqrt(2 * g * np.maximum(h_analytisch(t), 0.0))

t = np.linspace(0, T_E, 400)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(t, h_analytisch(t), 'b')
ax[0].set_xlabel('t [s]'); ax[0].set_ylabel('h(t) [m]')
ax[0].set_title('Spiegel h(t): Parabel')
ax[1].plot(t, Q_analytisch(t)*1000, 'r')
ax[1].set_xlabel('t [s]'); ax[1].set_ylabel('Q(t) [L/s]')
ax[1].set_title('Abfluss Q(t): linearer Abfall')
plt.tight_layout(); plt.show()

## 3. Explizites Euler-Verfahren

Idee: Zeit in Schritten $\Delta t$ zerlegen und die Ableitung durch eine Differenz ersetzen:

$$h_{n+1} = h_n + \Delta t \cdot f(t_n, h_n), \qquad f(t,h) = -\kappa\sqrt{h}$$

Physikalisch: **Momentaufnahmen** – in jedem Schritt wird der Zustand als quasistationär behandelt (genau der QS-Ansatz der Vorlesung, Folie 26). Kleines $\Delta t$ = Spiegel ändert sich pro Bild kaum = Quasistationarität gut erfüllt.

In [ ]:
def euler(dt, t_max=None):
    """Explizites Euler-Verfahren für A dh/dt = -alpha*A_o*sqrt(2gh).
    Rückgabe: Arrays t, h (Abbruch, sobald h <= 0).

    TODO (Aufgabe 5a):
      1. Berechnen Sie die rechte Seite f = -kappa*sqrt(h).
      2. Euler-Schritt: h_neu = h_alt + dt * f.
      3. Brechen Sie ab, sobald h <= 0 (h kann nicht negativ werden).
    """
    raise NotImplementedError("Aufgabe 5a: Erst die TODOs in euler() umsetzen!")

In [ ]:
# Vergleich numerisch vs. analytisch (dt = 10 s)
t_num, h_num = euler(dt=10.0)
t_ref = np.linspace(0, T_E, 400)

plt.figure(figsize=(8, 4))
plt.plot(t_ref, h_analytisch(t_ref), 'b-', label='analytisch')
plt.plot(t_num, h_num, 'ro-', markersize=3, label='Euler (dt = 10 s)')
plt.xlabel('t [s]'); plt.ylabel('h [m]')
plt.legend(); plt.grid(alpha=0.3)
plt.title('Behälterleerung: Euler vs. analytisch')
plt.show()

## 4. Schrittweitenstudie (Aufgabe 5b)

Vergleichen Sie $T_{E,num}$ mit dem analytischen Wert für $\Delta t = 100$ s, $10$ s, $1$ s, $0{,}1$ s.

In [ ]:
print(f"{'dt [s]':>8} | {'T_E,num [s]':>12} | {'Fehler [%]':>10}")
print('-' * 40)
for dt in [100.0, 10.0, 1.0, 0.1]:
    t_num, h_num = euler(dt)
    T_E_num = t_num[-1]
    fehler = (T_E_num - T_E) / T_E * 100
    print(f"{dt:8.1f} | {T_E_num:12.1f} | {fehler:+10.2f}")

In [ ]:
# Zoom ans Ende: Was passiert nahe h -> 0 bei grobem dt?
plt.figure(figsize=(8, 4))
t_ref = np.linspace(0.75*T_E, 1.05*T_E, 200)
plt.plot(t_ref, h_analytisch(np.minimum(t_ref, T_E)), 'b-', label='analytisch')
for dt, farbe in [(100.0, 'orange'), (10.0, 'green')]:
    t_num, h_num = euler(dt)
    plt.plot(t_num, h_num, 'o-', color=farbe, markersize=4, label=f'Euler (dt = {dt:g} s)')
plt.xlabel('t [s]'); plt.ylabel('h [m]')
plt.xlim(0.75*T_E, 1.05*T_E); plt.ylim(-0.15, 0.4)
plt.legend(); plt.grid(alpha=0.3)
plt.title('Nahe h -> 0: Euler schneidet die Parabel ab')
plt.show()

## 5. Vergleich mit adaptiver Methode (scipy)

`scipy.integrate.solve_ivp` wählt die Schrittweite automatisch (RK45). Für eine Vergleichsrechnung:

In [ ]:
from scipy.integrate import solve_ivp

def rhs(t, y):
    return [-kappa * np.sqrt(max(y[0], 0.0))]

event_leer = lambda t, y: y[0]            # Event: h = 0
event_leer.terminal = True
event_leer.direction = -1

t_span = (0.0, 3*T_E)
sol = solve_ivp(rhs, t_span, [h0], events=event_leer, rtol=1e-8, atol=1e-10)
T_E_adaptiv = sol.t_events[0][0]

print(f"T_E (solve_ivp, RK45) = {T_E_adaptiv:.2f} s")
print(f"T_E (analytisch)      = {T_E:.2f} s")
print(f"Abweichung            = {(T_E_adaptiv-T_E)/T_E*100:+.4f} %")

## 6. Interpretation (Aufgabe 5c) – in eigenen Worten beantworten

1. Was entspricht die Schrittweite $\Delta t$ im quasi-stationären Bild der Vorlesung?
2. Warum entleert die Euler-Lösung **systematisch zu schnell**? (Hinweis: Ist $h(t)$ konvex oder konkav? Wo liegen Tangenten an eine konvexe Kurve?)
3. Warum scheitert Euler nahe $h \to 0$ bei grobem $\Delta t$ – und was hat das mit $f(h) = -\kappa\sqrt{h}$ zu tun?
4. Übertragen Sie das Kriterium „kleines $\Delta t$" auf die Frage der Vorlesung: *Wann darf man quasi-stationär rechnen?* (Folie 26/36)